In [1]:
import requests
import os
import pandas as pd
import time
from datetime import datetime, timedelta
import gzip
import shutil

base_url = os.environ['CD2_BASE_URL']
client_id = os.environ['CD2_CLIENT_ID']
client_secret = os.environ['CD2_CLIENT_SECRET']

auth_url = f"{base_url}/ids/auth/login"
payload={'grant_type': 'client_credentials'}


In [2]:
r = requests.post(
    auth_url, 
    data=payload, 
    auth=(client_id, client_secret))
if r.status_code == 200:
    respons = r.json()
    access_token = respons['access_token']
    print("Henta access_token OK")
else:
    print(f"Klarte ikkje å skaffe access_token, feil {r.status_code}")

Henta access_token OK


In [3]:
def hent_filar(innfil, n):
    requesturl = f"{base_url}/dap/object/url"
    payload = f"{respons2['objects']}"
    payload = payload.replace('\'', '\"')
    headers = {'x-instauth': access_token, 'Content-Type': 'text/plain'}
    print(f"Hentar datafil nr. {n}, ", end="")
    r4 = requests.request("POST",requesturl, headers=headers, data=payload)
    if r4.status_code == 200:
        # print(f"Vellukka spørjing på {requesturl}")
        respons4 = r4.json()
        url = respons4['urls'][innfil]['url']
        data = requests.request("GET", url)
        no = datetime.now()
        utfil = f"{tabell}-{no.year}{no.month:02}{no.day:02}{no.hour:02}{no.minute:02}-{n}"
        open(f'{utfil}.gz', 'wb').write(data.content)
        with gzip.open(f'{utfil}.gz', 'rb') as f_in:
            with open(f'{utfil}.txt', 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        print(f" skrevet til {f'{utfil}.txt'}")
        # Fjern .gz-fila etter utpakking
        os.remove(f"{utfil}.gz")
    return f"{utfil}.txt"

# 1. Hente data

In [4]:
tabell = "enrollments"
no = datetime.now()
tidsrom = timedelta(hours=4)
sist_oppdatert = (no - tidsrom).isoformat(timespec='seconds') + "Z"
# with open(f"sist_oppdatert_{tabell}.txt", "r") as f_in:
#     sist_oppdatert = f_in.read()
# sist_oppdatert = "2025-09-29T11:27:05Z"
requesturl = f"{base_url}/dap/query/canvas/table/{tabell}/data"
payload = '{"format": "csv", "since": \"%s\"}' %(sist_oppdatert)
headers = {'x-instauth': access_token, 'Content-Type': 'text/plain'}
print(f"Sender søk til {requesturl}")
r = requests.request("POST", requesturl, headers=headers, data=payload)
if 200 <= r.status_code < 300:
    respons = r.json()
    id = respons['id']
    print(f"Sjekker status på jobb {id}")
    les_data = True
    while les_data:
        requesturl = f"{base_url}/dap//job/{id}"
        r2 = requests.request("GET", requesturl, headers=headers)
        respons2 = r2.json()
        print(respons2)
        if respons2['status'] == "complete":
            les_data = False
        time.sleep(5)
else:
    print(f"Feil i spørsmål, kode {respons}")
antal = len(respons2['objects'])
for i in range(antal):
    utfil = hent_filar(respons2['objects'][i]['id'], i)

Sender søk til https://api-gateway.instructure.com/dap/query/canvas/table/enrollments/data
Sjekker status på jobb 42edf260-f24e-4487-9749-22e39a60ac5b
{'id': '42edf260-f24e-4487-9749-22e39a60ac5b', 'status': 'running', 'expires_at': '2025-10-01T05:42:00Z'}
{'id': '42edf260-f24e-4487-9749-22e39a60ac5b', 'status': 'running', 'expires_at': '2025-10-01T05:42:00Z'}
{'id': '42edf260-f24e-4487-9749-22e39a60ac5b', 'status': 'running', 'expires_at': '2025-10-01T05:42:00Z'}
{'id': '42edf260-f24e-4487-9749-22e39a60ac5b', 'status': 'running', 'expires_at': '2025-10-01T05:42:00Z'}
{'id': '42edf260-f24e-4487-9749-22e39a60ac5b', 'status': 'running', 'expires_at': '2025-10-01T05:42:00Z'}
{'id': '42edf260-f24e-4487-9749-22e39a60ac5b', 'status': 'running', 'expires_at': '2025-10-01T05:42:00Z'}
{'id': '42edf260-f24e-4487-9749-22e39a60ac5b', 'status': 'running', 'expires_at': '2025-10-01T05:42:00Z'}
{'id': '42edf260-f24e-4487-9749-22e39a60ac5b', 'status': 'running', 'expires_at': '2025-10-01T05:42:00Z'}
{

## 2. Lese inn og analysere data

In [ ]:
innfil = utfil
with open(innfil, "r", encoding="utf-8") as f_in:
    linjer = f_in.readlines()
headings = linjer[0].split(",")
data = [line.strip().split(",") for line in linjer[1:]]
df = pd.DataFrame(data, columns=headings)

['key.id', 'value.sis_batch_id', 'value.user_id', 'value.created_at', 'value.updated_at', 'value.workflow_state', 'value.role_id', 'value.start_at', 'value.end_at', 'value.course_id', 'value.completed_at', 'value.course_section_id', 'value.grade_publishing_status', 'value.associated_user_id', 'value.self_enrolled', 'value.limit_privileges_to_course_section', 'value.last_activity_at', 'value.total_activity_time', 'value.sis_pseudonym_id', 'value.last_attended_at', 'value.type', 'meta.ts', 'meta.action\n']


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24569 entries, 0 to 24568
Data columns (total 23 columns):
 #   Column                                    Non-Null Count  Dtype 
---  ------                                    --------------  ----- 
 0   key.id                                    24569 non-null  object
 1   value.sis_batch_id                        24569 non-null  object
 2   value.user_id                             24569 non-null  object
 3   value.created_at                          24569 non-null  object
 4   value.updated_at                          24569 non-null  object
 5   value.workflow_state                      24569 non-null  object
 6   value.role_id                             24569 non-null  object
 7   value.start_at                            24569 non-null  object
 8   value.end_at                              24569 non-null  object
 9   value.course_id                           24569 non-null  object
 10  value.completed_at                        2456